# Подготовка данных для PostgreSQL

В этом ноутбуке я проверю связи между сущностями и подготовлю отдельные таблицы для загрузки в PostgreSQL.

In [1]:
import pandas as pd

pd.set_option('display.max_columns', None)

## Загрузка очищенных данных

In [2]:
df = pd.read_pickle('online_retail_clean.pkl')
df.shape

(1055238, 13)

## Проверка ключей

Перед разделением данных проверим, соответствует ли одному счёту один покупатель, одна дата и одна страна.

In [3]:
invoice_check = df.groupby('invoice').agg(customers=('customer_id', 'nunique'), dates=('invoice_date', 'nunique'), countries=('country', 'nunique')).reset_index()
invoice_check[['customers', 'dates', 'countries']].max()

customers    1
dates        2
countries    1
dtype: int64

In [4]:
print('Счета с несколькими покупателями:', (invoice_check['customers'] > 1).sum())
print('Счета с несколькими датами:', (invoice_check['dates'] > 1).sum())
print('Счета с несколькими странами:', (invoice_check['countries'] > 1).sum())

Счета с несколькими покупателями: 0
Счета с несколькими датами: 83
Счета с несколькими странами: 0


### Счета с несколькими датами

У части счетов позиции имеют разные значения `invoice_date`. Посмотрим несколько таких случаев.

In [5]:
multiple_dates = invoice_check[invoice_check['dates'] > 1]['invoice']

df[df['invoice'].isin(multiple_dates)][['invoice', 'invoice_date']].drop_duplicates().sort_values(['invoice', 'invoice_date']).head(30)

,invoice,invoice_date
41630,492807,2009-12-20 12:28:00
41655,492807,2009-12-20 12:29:00
52310,494166,2010-01-12 09:47:00
52516,494166,2010-01-12 09:48:00
110024,499967,2010-03-03 14:06:00
110052,499967,2010-03-03 14:07:00
115209,500353,2010-03-07 15:24:00
115232,500353,2010-03-07 15:25:00
119364,500827,2010-03-10 11:10:00
119527,500827,2010-03-10 11:11:00


In [6]:
invoice_dates = df.groupby('invoice')['invoice_date'].agg(['min', 'max'])
invoice_dates['diff_minutes'] = (invoice_dates['max'] - invoice_dates['min']).dt.total_seconds() / 60

invoice_dates[invoice_dates['diff_minutes'] > 0]['diff_minutes'].describe()

count    83.000000
mean      1.144578
std       0.938766
min       1.000000
25%       1.000000
50%       1.000000
75%       1.000000
max       9.000000
Name: diff_minutes, dtype: float64

In [7]:
invoice_dates[invoice_dates['diff_minutes'] > 0].sort_values('diff_minutes', ascending=False).head(10)

,min,max,diff_minutes
invoice,,,
529368,2010-10-28 10:07:00,2010-10-28 10:16:00,9.0
520878,2010-08-31 15:32:00,2010-08-31 15:36:00,4.0
521901,2010-09-09 12:35:00,2010-09-09 12:37:00,2.0
547690,2011-03-24 14:55:00,2011-03-24 14:56:00,1.0
546986,2011-03-18 12:55:00,2011-03-18 12:56:00,1.0
546388,2011-03-11 13:42:00,2011-03-11 13:43:00,1.0
545713,2011-03-07 10:11:00,2011-03-07 10:12:00,1.0
545460,2011-03-02 17:32:00,2011-03-02 17:33:00,1.0
544926,2011-02-24 17:50:00,2011-02-24 17:51:00,1.0


У 83 счетов время отдельных позиций отличается. Максимальная разница составляет 9 минут, а в большинстве случаев — 1 минуту.

Будем считать временем счёта минимальное `invoice_date`, то есть момент появления первой позиции.

Проверим, встречается ли один покупатель в нескольких странах.

In [8]:
customer_country_check = df[df['customer_id'].notna()].groupby('customer_id')['country'].nunique()

print('Покупателей с несколькими странами:', (customer_country_check > 1).sum())
print('Максимальное количество стран у покупателя:', customer_country_check.max())

Покупателей с несколькими странами: 13
Максимальное количество стран у покупателя: 2


Проверим, сколько разных описаний может соответствовать одному коду товара.

In [9]:
product_description_check = df[df['description'].notna()].groupby('stock_code')['description'].nunique()

print('Товаров с несколькими описаниями:', (product_description_check > 1).sum())
print('Максимальное количество описаний:', product_description_check.max())

Товаров с несколькими описаниями: 1213
Максимальное количество описаний: 9


In [10]:
product_description_check.sort_values(ascending=False).head(10)

stock_code
20713     9
21181     7
22734     7
22423     7
23084     7
47566B    6
21830     6
85175     6
22501     5
85172     5
Name: description, dtype: int64

У части товаров одному `stock_code` соответствует несколько описаний. Проверим несколько таких товаров и выберем правило для формирования справочника товаров.

In [11]:
df[df['stock_code'] == '20713'][['stock_code', 'description']].drop_duplicates().sort_values('description')

,stock_code,description
939606,20713,Found
834,20713,JUMBO BAG OWLS
945852,20713,Marked as 23343
928995,20713,found
261489,20713,missing
948211,20713,wrongly coded 23343
906149,20713,wrongly coded-23343
941043,20713,wrongly marked 23343
789345,20713,wrongly marked. 23343 in box
318908,20713,NaN


Для справочника товаров в качестве основного описания возьмём наиболее часто встречающееся описание каждого `stock_code`. Остальные варианты считаем альтернативными или ошибочными написаниями.

In [12]:
all_products = df[['stock_code']].drop_duplicates()
product_names = df.dropna(subset=['description']).groupby(['stock_code', 'description']).size().reset_index(name='count')
main_names = product_names.sort_values(['stock_code', 'count'], ascending=[True, False]).drop_duplicates('stock_code')[['stock_code', 'description']]
products = all_products.merge(main_names, on='stock_code', how='left').reset_index(drop=True)

products.head()

,stock_code,description
0,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS
1,79323P,PINK CHERRY LIGHTS
2,79323W,WHITE CHERRY LIGHTS
3,22041,"RECORD FRAME 7"" SINGLE SIZE"
4,21232,STRAWBERRY CERAMIC TRINKET BOX


In [13]:
print('Товаров:', len(products))
print('Уникальных stock_code:', products['stock_code'].nunique())
print('Товаров без описания:', products['description'].isna().sum())

Товаров: 5304
Уникальных stock_code: 5304
Товаров без описания: 355


## Формирование таблиц

Разделим исходные транзакции на покупателей, товары, счета и позиции счетов.

In [14]:
customers = df[df['customer_id'].notna()][['customer_id']].drop_duplicates().reset_index(drop=True)

customers.head()

,customer_id
0,13085
1,13078
2,15362
3,18102
4,12682


In [15]:
print('Покупателей:', len(customers))
print('Уникальных customer_id:', customers['customer_id'].nunique())

Покупателей: 5942
Уникальных customer_id: 5942


In [16]:
products.head()

,stock_code,description
0,85048,15CM CHRISTMAS GLASS BALL 20 LIGHTS
1,79323P,PINK CHERRY LIGHTS
2,79323W,WHITE CHERRY LIGHTS
3,22041,"RECORD FRAME 7"" SINGLE SIZE"
4,21232,STRAWBERRY CERAMIC TRINKET BOX


In [17]:
orders = df.groupby('invoice', as_index=False).agg(customer_id=('customer_id', 'first'), invoice_date=('invoice_date', 'min'), country=('country', 'first'), is_cancelled=('is_cancelled', 'first'))

orders.head()

,invoice,customer_id,invoice_date,country,is_cancelled
0,489434,13085,2009-12-01 07:45:00,United Kingdom,False
1,489435,13085,2009-12-01 07:46:00,United Kingdom,False
2,489436,13078,2009-12-01 09:06:00,United Kingdom,False
3,489437,15362,2009-12-01 09:08:00,United Kingdom,False
4,489438,18102,2009-12-01 09:24:00,United Kingdom,False


In [18]:
print('Счетов:', len(orders))
print('Уникальных invoice:', orders['invoice'].nunique())

Счетов: 53628
Уникальных invoice: 53628


In [19]:
order_items = df[['invoice', 'stock_code', 'quantity', 'price', 'line_total']].copy()

order_items.head()

,invoice,stock_code,quantity,price,line_total
0,489434,85048,12,6.95,83.4
1,489434,79323P,12,6.75,81.0
2,489434,79323W,12,6.75,81.0
3,489434,22041,48,2.10,100.8
4,489434,21232,24,1.25,30.0


In [20]:
order_items.insert(0, 'order_item_id', range(1, len(order_items) + 1))

In [21]:
print('Позиций:', len(order_items))
print('Уникальных order_item_id:', order_items['order_item_id'].nunique())

Позиций: 1055238
Уникальных order_item_id: 1055238


In [22]:
missing_orders = (~order_items['invoice'].isin(orders['invoice'])).sum()
print('Позиций без счёта:', missing_orders)

Позиций без счёта: 0


In [23]:
missing_products = (~order_items['stock_code'].isin(products['stock_code'])).sum()
print('Позиций без товара:', missing_products)

Позиций без товара: 0


In [24]:
known_customers = orders['customer_id'].dropna()
missing_customers = (~known_customers.isin(customers['customer_id'])).sum()
print('Счетов с неизвестным customer_id:', missing_customers)

Счетов с неизвестным customer_id: 0


## Проверка связей между таблицами

Перед сохранением данных проверим, что все внешние ключи имеют соответствующие записи в родительских таблицах.

In [25]:
print('Позиций без счёта:', (~order_items['invoice'].isin(orders['invoice'])).sum())
print('Позиций без товара:', (~order_items['stock_code'].isin(products['stock_code'])).sum())

known_customers = orders['customer_id'].dropna()
print('Счетов с неизвестным покупателем:', (~known_customers.isin(customers['customer_id'])).sum())

Позиций без счёта: 0
Позиций без товара: 0
Счетов с неизвестным покупателем: 0


In [26]:
print('customers:', customers.shape)
print('products:', products.shape)
print('orders:', orders.shape)
print('order_items:', order_items.shape)

customers: (5942, 1)
products: (5304, 2)
orders: (53628, 5)
order_items: (1055238, 6)


## Сохранение таблиц

После проверки связей сохраним подготовленные таблицы в CSV для загрузки в PostgreSQL.

In [27]:
customers.to_csv('customers.csv', index=False)
products.to_csv('products.csv', index=False)
orders.to_csv('orders.csv', index=False)
order_items.to_csv('order_items.csv', index=False)

In [28]:
print('customers:', len(customers))
print('products:', len(products))
print('orders:', len(orders))
print('order_items:', len(order_items))
print('CSV-файлы сохранены')

customers: 5942
products: 5304
orders: 53628
order_items: 1055238
CSV-файлы сохранены


## Итог

Исходная транзакционная таблица была разделена на четыре связанные сущности: покупателей, товары, счета и позиции счетов.

Для счетов, у которых позиции имели разное время, в качестве `invoice_date` использовано самое раннее время. Для каждого товара выбрано наиболее часто встречающееся описание, а товары без описания сохранены с пустым значением.

Перед сохранением были проверены связи между таблицами. Все значения внешних ключей имеют соответствующие записи в родительских таблицах.

Подготовленные CSV-файлы можно использовать для загрузки в PostgreSQL.